# Week 7: Autoencoder Embeddings vs PCA

This notebook runs a fast dimensionality-reduction experiment on the same engineered Week 5 feature matrix used for PCA.

It trains a Keras autoencoder with latent dimensions from 2 to 30, compares reconstruction quality against PCA at the same dimension, and saves the best embedding for downstream K-means.

Outputs written to `artifacts/week07/`:
- sweep metrics table
- comparison plots
- best autoencoder embeddings
- best-model reconstruction summary

## Model design

The autoencoder is intentionally compact but regularized so it behaves well on a tabular movie feature matrix:

- Gaussian noise at the input for denoising
- dense encoder blocks with batch normalization and swish activations
- dropout and light L2 regularization to reduce overfitting
- symmetric decoder with a linear output layer
- early stopping and learning-rate reduction during training

This makes the experiment fast enough for a Week 7 sweep while still being more expressive than PCA.

In [1]:
!pip install keras


[notice] A new release of pip is available: 26.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [2]:
from pathlib import Path

import json
import time

import numpy as np
import pandas as pd
import polars as pl
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display
from sklearn.decomposition import PCA
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

try:
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers, regularizers
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        'TensorFlow/Keras is required for this notebook. Install it with `pip install tensorflow-cpu` or add it to your environment.'
    ) from exc

keras.utils.set_random_seed(42)
try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass

print(f'TensorFlow version: {tf.__version__}')
print(f'Built with CUDA: {tf.test.is_built_with_cuda()}')
print(f'Physical GPUs: {tf.config.list_physical_devices("GPU")}')
print(f'Logical GPUs: {tf.config.list_logical_devices("GPU")}')
if tf.config.list_physical_devices('GPU'):
    print('Status: TensorFlow can see a GPU.')
else:
    print('Status: TensorFlow is running on CPU.')

project_root = Path.cwd()
if not (project_root / 'data').exists():
    project_root = project_root.parent
if not (project_root / 'data').exists():
    project_root = project_root.parent

ARTIFACTS_DIR = project_root / 'artifacts' / 'week07'
WEEK05_ARTIFACTS = project_root / 'artifacts' / 'week05'
WEEK05_PROCESSED = project_root / 'data' / 'processed' / 'week05'

ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

input_path = WEEK05_ARTIFACTS / 'week05_pca_feature_matrix.parquet'
if not input_path.exists():
    raise FileNotFoundError(f'Could not find the Week 5 PCA feature matrix at {input_path}.')

input_path

TensorFlow version: 2.16.2
Built with CUDA: False
Physical GPUs: []
Logical GPUs: []
Status: TensorFlow is running on CPU.


PosixPath('/Users/jay17/Documents/Proyects/big-data-tf/artifacts/week05/week05_pca_feature_matrix.parquet')

## Load the Week 5 engineered feature matrix

We use the PCA-ready Week 5 feature matrix so the autoencoder and PCA are compared on the same input space.

In [3]:
feature_frame = pl.read_parquet(input_path)
if 'movieId' not in feature_frame.columns:
    raise ValueError('Expected a movieId column in the Week 5 PCA feature matrix.')

feature_columns = [
    'rating_std',
    'release_year_z',
    'avg_rating_z',
    'rating_count_log',
    'tag_event_count_log',
    'unique_tag_count_log',
    'rating_span_seconds_z',
    'tag_span_seconds_z',
    'genre_count_log',
    'genre_drama',
    'genre_comedy',
    'genre_thriller',
    'genre_romance',
    'genre_action',
    'genre_horror',
    'genre_documentary',
    'genre_crime',
    'genre_adventure',
    'genre_sci_fi',
    'genre_children',
    'genre_animation',
    'genre_mystery',
    'genre_fantasy',
    'genre_war',
    'genre_western',
    'genre_musical',
    'genre_film_noir',
    'genre_imax',
    'tag_000_bd_r',
    'tag_001_woman_director',
    'tag_002_murder',
    'tag_003_independent_film',
    'tag_004_comedy',
    'tag_005_nudity_topless',
    'tag_006_based_on_a_book',
    'tag_007_clv',
    'tag_008_drama',
    'tag_009_romance',
    'tag_010_funny',
    'tag_011_violence',
    'tag_012_based_on_novel_or_book',
    'tag_013_revenge',
    'tag_014_musical',
    'tag_015_criterion',
    'tag_016_betamax',
    'tag_017_love',
    'tag_018_family',
    'tag_019_action',
]

missing_columns = [column for column in feature_columns if column not in feature_frame.columns]
if missing_columns:
    raise ValueError(f'Missing expected PCA feature columns: {missing_columns}')

feature_df = feature_frame.select(feature_columns).to_pandas().astype(float)
movie_ids = feature_frame.get_column('movieId').to_list()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(feature_df)
X_train, X_val = train_test_split(X_scaled, test_size=0.2, random_state=42)

print(f'Loaded {feature_frame.height:,} movies x {len(feature_columns)} PCA features from {input_path}')
print(f'Train shape: {X_train.shape}, Validation shape: {X_val.shape}')
display(feature_frame.select(['movieId'] + feature_columns[:6]).head(5).to_pandas())

Loaded 62,423 movies x 48 PCA features from /Users/jay17/Documents/Proyects/big-data-tf/artifacts/week05/week05_pca_feature_matrix.parquet
Train shape: (49938, 48), Validation shape: (12485, 48)


,movieId,rating_std,release_year_z,avg_rating_z,rating_count_log,tag_event_count_log,unique_tag_count_log
0,1,0.921552,0.113798,1.111502,10.956230,6.548219,4.770685
1,2,0.959851,0.113798,0.243503,10.095306,5.198497,3.761200
2,3,1.008443,0.113798,0.095499,9.376278,3.401197,3.135494
3,4,1.108531,0.113798,-0.294424,7.833600,2.484907,2.197225
4,5,0.996611,0.113798,-0.017490,9.368625,3.218876,2.995732


## Autoencoder architecture

The encoder is deeper than a minimal bottleneck MLP, but still compact enough to train quickly on a tabular feature matrix.

The latent layer is the only part that changes across the sweep.

In [4]:
def build_encoder_block(x, units, dropout_rate=0.0):
    """Single encoder block: Dense -> BatchNorm -> Swish -> Dropout"""
    x = layers.Dense(units, kernel_initializer='he_normal', kernel_regularizer=regularizers.l2(1e-5), use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('swish')(x)
    if dropout_rate > 0:
        x = layers.Dropout(dropout_rate)(x)
    return x

def build_autoencoder(input_dim: int, latent_dim: int):
    """Build a symmetric autoencoder with encoder blocks and decoder mirror."""
    inputs = keras.Input(shape=(input_dim,), name='features')

    # Encoder: 256 → 128 → 64 → latent_dim
    x = layers.GaussianNoise(0.02)(inputs)
    x = build_encoder_block(x, 256, dropout_rate=0.15)
    x = build_encoder_block(x, 128, dropout_rate=0.10)
    x = build_encoder_block(x, 64)
    embedding = layers.Dense(latent_dim, name='embedding')(x)

    # Decoder: latent_dim → 64 → 128 → 256 → input_dim
    x = build_encoder_block(embedding, 64)
    x = build_encoder_block(x, 128)
    x = build_encoder_block(x, 256)
    outputs = layers.Dense(input_dim, activation='linear', name='reconstruction')(x)

    autoencoder = keras.Model(inputs, outputs, name=f'autoencoder_latent_{latent_dim}')
    encoder = keras.Model(inputs, embedding, name=f'encoder_latent_{latent_dim}')

    autoencoder.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3), loss='mse')
    return autoencoder, encoder

def fit_autoencoder(latent_dim: int, X_train: np.ndarray, X_val: np.ndarray):
    """Train autoencoder with early stopping and learning-rate reduction."""
    autoencoder, encoder = build_autoencoder(X_train.shape[1], latent_dim)
    callbacks = [
        keras.callbacks.EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True, min_delta=1e-5),
        keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, min_lr=1e-5, verbose=0),
    ]
    start = time.time()
    history = autoencoder.fit(X_train, X_train, validation_data=(X_val, X_val), epochs=60, batch_size=128, shuffle=True, verbose=1, callbacks=callbacks)
    return autoencoder, encoder, history, time.time() - start

def pca_reconstruction_mse(X_train: np.ndarray, X_val: np.ndarray, latent_dim: int):
    """Fit PCA and compute validation reconstruction error."""
    pca = PCA(n_components=latent_dim, random_state=42)
    pca.fit(X_train)
    reconstructed = pca.inverse_transform(pca.transform(X_val))
    return float(mean_squared_error(X_val, reconstructed)), pca

## Sweep latent dimensions 2..20

For each latent dimension, this cell trains one autoencoder and computes its validation reconstruction error.

The same latent dimension is evaluated with PCA so the comparison is apples-to-apples.

In [5]:
latent_dims = list(range(2, 31))
results = []
best_run = None
best_val_mse = np.inf

for latent_dim in latent_dims:
    print(f'\n=== Training latent_dim={latent_dim} ===')
    autoencoder, encoder, history, elapsed = fit_autoencoder(latent_dim, X_train, X_val)
    ae_val_pred = autoencoder.predict(X_val, verbose=0)
    ae_train_pred = autoencoder.predict(X_train, verbose=0)
    ae_val_mse = float(mean_squared_error(X_val, ae_val_pred))
    ae_train_mse = float(mean_squared_error(X_train, ae_train_pred))
    pca_val_mse, pca_model = pca_reconstruction_mse(X_train, X_val, latent_dim)

    row = {
        'latent_dim': latent_dim,
        'ae_train_mse': ae_train_mse,
        'ae_val_mse': ae_val_mse,
        'pca_val_mse': pca_val_mse,
        'ae_vs_pca_ratio': ae_val_mse / pca_val_mse if pca_val_mse else np.nan,
        'epochs_ran': len(history.history['loss']),
        'final_train_loss': float(history.history['loss'][-1]),
        'final_val_loss': float(history.history['val_loss'][-1]),
        'training_seconds': float(elapsed),
    }
    results.append(row)

    if ae_val_mse < best_val_mse:
        best_val_mse = ae_val_mse
        best_run = {
            'latent_dim': latent_dim,
            'autoencoder': autoencoder,
            'encoder': encoder,
            'pca_model': pca_model,
            'metrics': row,
        }

sweep_df = pd.DataFrame(results)
sweep_df['ae_improvement_over_pca'] = sweep_df['pca_val_mse'] - sweep_df['ae_val_mse']

sweep_path = ARTIFACTS_DIR / 'week07_autoencoder_vs_pca_sweep.csv'
sweep_df.to_csv(sweep_path, index=False)

display(sweep_df)
print(f'Saved sweep table to {sweep_path}')
print(f'Best latent dimension by autoencoder validation MSE: {best_run["latent_dim"]}')


=== Training latent_dim=2 ===
Epoch 1/60
391/391 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.7955 - val_loss: 0.7045 - learning_rate: 0.0010
Epoch 2/60
391/391 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.6962 - val_loss: 0.6354 - learning_rate: 0.0010
Epoch 3/60
391/391 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.6505 - val_loss: 0.6050 - learning_rate: 0.0010
Epoch 4/60
391/391 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.6247 - val_loss: 0.5842 - learning_rate: 0.0010
Epoch 5/60
391/391 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.6055 - val_loss: 0.5671 - learning_rate: 0.0010
Epoch 6/60
391/391 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.5896 - val_loss: 0.5539 - learning_rate: 0.0010
Epoch 7/60
391/391 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.5768 - val_loss: 0.5405 - learning_rate: 0.0010
Epoch 8/60
391/391 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.5668 - val_loss: 0.5338 - learning_rate: 0.0010
Epoch 9/60
391/391 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.5571 - val_loss: 0.5205 - l

,latent_dim,ae_train_mse,ae_val_mse,pca_val_mse,ae_vs_pca_ratio,epochs_ran,final_train_loss,final_val_loss,training_seconds,ae_improvement_over_pca
0,2,0.400938,0.404569,0.831296,0.486672,60,0.442174,0.418342,43.497694,0.426727
1,3,0.256233,0.272022,0.788445,0.345011,60,0.310143,0.288112,44.534954,0.516422
2,4,0.194705,0.212029,0.748629,0.283223,60,0.247144,0.229165,51.669859,0.536600
3,5,0.149473,0.169822,0.715331,0.237404,60,0.203540,0.185767,54.897359,0.545508
4,6,0.116970,0.133465,0.685206,0.194780,60,0.170844,0.149253,54.135497,0.551741
5,7,0.097435,0.112603,0.655701,0.171729,60,0.149340,0.126903,54.244104,0.543098
6,8,0.078696,0.095822,0.627857,0.152618,60,0.128579,0.109462,67.696259,0.532035
7,9,0.059261,0.073853,0.600650,0.122954,60,0.108106,0.086228,48.512121,0.526798
8,10,0.054226,0.066547,0.574777,0.115778,60,0.098808,0.079156,56.632957,0.508231
9,11,0.039980,0.053246,0.550940,0.096645,60,0.082930,0.064677,57.681533,0.497694


Saved sweep table to /Users/jay17/Documents/Proyects/big-data-tf/artifacts/week07/week07_autoencoder_vs_pca_sweep.csv
Best latent dimension by autoencoder validation MSE: 30


## Comparison plots

The first chart compares autoencoder and PCA reconstruction error across latent dimensions.

The second chart shows the improvement or gap between them.

In [17]:
fig_mse = go.Figure()

fig_mse.add_trace(
    go.Scatter(
        x=sweep_df['latent_dim'],
        y=sweep_df['ae_val_mse'],
        mode='lines+markers',
        name='Autoencoder',
        line=dict(color='#1f77b4', width=3),
    )
)

fig_mse.add_trace(
    go.Scatter(
        x=sweep_df['latent_dim'],
        y=sweep_df['pca_val_mse'],
        mode='lines+markers',
        name='PCA',
        line=dict(color='#d62728', width=3),
    )
)

fig_mse.update_layout(
    title='Validation Reconstruction MSE',
    xaxis_title='latent dimension',
    yaxis_title='reconstruction MSE',
    template='plotly_white',
    width=800,
    height=500,
)

mse_path = ARTIFACTS_DIR / 'week07_validation_mse.html'
fig_mse.write_html(str(mse_path))

try:
    fig_mse.write_image(str(mse_path.with_suffix('.png')))
except Exception:
    pass

fig_mse.show()


fig_gap = go.Figure()

fig_gap.add_trace(
    go.Bar(
        x=sweep_df['latent_dim'],
        y=sweep_df['ae_improvement_over_pca'],
        name='PCA MSE - AE MSE',
        marker_color='#2ca02c',
    )
)

fig_gap.update_layout(
    title='Autoencoder Advantage over PCA',
    xaxis_title='latent dimension',
    yaxis_title='MSE gap',
    template='plotly_white',
    width=800,
    height=500,
)

gap_path = ARTIFACTS_DIR / 'week07_ae_vs_pca_gap.html'
fig_gap.write_html(str(gap_path))

try:
    fig_gap.write_image(str(gap_path.with_suffix('.png')))
except Exception:
    pass

fig_gap.show()

print(f'Saved plots to:\n{mse_path}\n{gap_path}')

Saved plots to:
/Users/jay17/Documents/Proyects/big-data-tf/artifacts/week07/week07_validation_mse.html
/Users/jay17/Documents/Proyects/big-data-tf/artifacts/week07/week07_ae_vs_pca_gap.html


## Select best embedding dimension

Review the sweep table above and choose your preferred latent dimension.

The notebook found the minimum MSE at the highest dimension, but you may prefer a lower-dimensional embedding if it gives similar reconstruction quality.

**Decision rule:** Look for the "elbow" where MSE stops improving much, or pick the dimension where autoencoder beats PCA by a good margin at lower cost.

Edit the cell below to set your chosen dimension.

In [18]:
# ===== CHOOSE YOUR DIMENSION HERE =====
# Set chosen_latent_dim to the dimension you want from the sweep table above.
# The default is the minimum MSE (typically the highest dimension).
# But you can pick any dimension for better efficiency trade-offs.

# chosen_latent_dim = best_run['latent_dim']  # Change this to your preferred dimension
chosen_latent_dim = 13

# Show stats for your choice
chosen_metrics = sweep_df[sweep_df['latent_dim'] == chosen_latent_dim].iloc[0]
print(f"Selected latent_dim={chosen_latent_dim}")
print(f"  AE validation MSE: {chosen_metrics['ae_val_mse']:.6f}")
print(f"  PCA validation MSE: {chosen_metrics['pca_val_mse']:.6f}")
print(f"  AE advantage: {chosen_metrics['ae_improvement_over_pca']:.6f} (positive = AE wins)")


Selected latent_dim=13
  AE validation MSE: 0.034674
  PCA validation MSE: 0.508349
  AE advantage: 0.473675 (positive = AE wins)


In [16]:
best_dim = int(chosen_latent_dim)

# If you chose a different dimension than the auto-selected minimum MSE,
# we need to retrain that specific model to export the encoder.
# (The sweep loop only saved the minimum-MSE model.)
if best_dim == best_run['latent_dim']:
    best_encoder = best_run['encoder']
    best_autoencoder = best_run['autoencoder']
    print(f"Using pre-trained model from the sweep (already trained during loop).")
else:
    print(f"Training autoencoder for latent_dim={best_dim} for 120 epochs on train/val split...")
    best_autoencoder, best_encoder = build_autoencoder(X_train.shape[1], best_dim)
    best_autoencoder.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3), loss='mse')
    callbacks_pretrain = [
        keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, min_delta=1e-4),
        keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-5, verbose=0),
    ]
    start_pretrain = time.time()
    history_pretrain = best_autoencoder.fit(
        X_train, X_train,
        validation_data=(X_val, X_val),
        epochs=200,
        batch_size=128,
        shuffle=True,
        verbose=1,
        callbacks=callbacks_pretrain
    )
    elapsed_pretrain = time.time() - start_pretrain
    print(f'Pre-training completed in {elapsed_pretrain:.1f} seconds')

best_metrics = sweep_df[sweep_df['latent_dim'] == best_dim].iloc[0]

print(f'Loaded autoencoder for latent_dim={best_dim}')
print(f'  Sweep validation MSE: {best_metrics["ae_val_mse"]:.6f}')

Training autoencoder for latent_dim=13 for 120 epochs on train/val split...
Epoch 1/200
391/391 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.5177 - val_loss: 0.2751 - learning_rate: 0.0010
Epoch 2/200
391/391 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.2792 - val_loss: 0.1919 - learning_rate: 0.0010
Epoch 3/200
391/391 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.2307 - val_loss: 0.1608 - learning_rate: 0.0010
Epoch 4/200
391/391 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.2052 - val_loss: 0.1423 - learning_rate: 0.0010
Epoch 5/200
391/391 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.1880 - val_loss: 0.1286 - learning_rate: 0.0010
Epoch 6/200
391/391 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.1745 - val_loss: 0.1203 - learning_rate: 0.0010
Epoch 7/200
391/391 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.1644 - val_loss: 0.1132 - learning_rate: 0.0010
Epoch 8/200
391/391 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.1560 - val_loss: 0.1062 - learning_rate: 0.0010
Epoch 9/200
391/391 ━━━━━━━━━━━━━━━━

KeyboardInterrupt: 

In [ ]:
if best_dim == best_run['latent_dim']:
    print("(Skipping training plot—using pre-trained model from sweep)")
else:
    fig = go.Figure()

    fig.add_trace(go.Scatter(
        y=history_pretrain.history['loss'],
        mode='lines',
        name='Train Loss',
        line=dict(color='#1f77b4', width=2)
    ))

    fig.add_trace(go.Scatter(
        y=history_pretrain.history['val_loss'],
        mode='lines',
        name='Validation Loss',
        line=dict(color='#d62728', width=2)
    ))

    fig.update_layout(
        title=f'Training history for latent_dim={best_dim} (200 epochs)',
        xaxis_title='Epoch',
        yaxis_title='Loss (MSE)',
        template='plotly_white',
        height=500,
        width=900,
        hovermode='x unified'
    )

    fig.show()

In [ ]:
# Extract encoder and generate embeddings
best_encoder = keras.Model(best_autoencoder.input, best_autoencoder.get_layer('embedding').output)

full_embeddings = best_encoder.predict(X_scaled, verbose=0)
embedding_columns = [f'ae_{idx + 1}' for idx in range(full_embeddings.shape[1])]
embedding_frame = pd.DataFrame(full_embeddings, columns=embedding_columns)
embedding_frame.insert(0, 'movieId', movie_ids)

embedding_path = ARTIFACTS_DIR / f'week07_autoencoder_embeddings_latent_{best_dim}.parquet'
pl.from_pandas(embedding_frame).write_parquet(embedding_path)

best_reconstruction = best_autoencoder.predict(X_scaled, verbose=0)
best_reconstruction_mse = float(mean_squared_error(X_scaled, best_reconstruction))

summary = {
    'input_path': str(input_path),
    'n_rows': int(feature_frame.height),
    'n_features': int(len(feature_columns)),
    'chosen_latent_dim': best_dim,
    'chosen_ae_validation_mse': float(best_metrics['ae_val_mse']),
    'chosen_pca_validation_mse': float(best_metrics['pca_val_mse']),
    'chosen_ae_vs_pca_ratio': float(best_metrics['ae_vs_pca_ratio']),
    'chosen_full_data_reconstruction_mse': best_reconstruction_mse,
}

summary_path = ARTIFACTS_DIR / 'week07_autoencoder_summary.json'
summary_path.write_text(json.dumps(summary, indent=2))

print(f'\n=== Export from trained model ===')
print(f'Chosen latent dimension: {best_dim}')
print(f'Chosen AE validation MSE: {best_metrics["ae_val_mse"]:.6f}')
print(f'Chosen PCA validation MSE: {best_metrics["pca_val_mse"]:.6f}')
print(f'Full-data reconstruction MSE: {best_reconstruction_mse:.6f}')
print(f'Saved embeddings to {embedding_path}')
print(f'Saved summary to {summary_path}')
display(embedding_frame.head(5))


=== Export from trained model ===
Chosen latent dimension: 13
Chosen AE validation MSE: 0.034674
Chosen PCA validation MSE: 0.508349
Full-data reconstruction MSE: 0.018395
Saved embeddings to /Users/jay17/Documents/Proyects/big-data-tf/artifacts/week07/week07_autoencoder_embeddings_latent_13.parquet
Saved summary to /Users/jay17/Documents/Proyects/big-data-tf/artifacts/week07/week07_autoencoder_summary.json


,movieId,ae_1,ae_2,ae_3,ae_4,ae_5,ae_6,ae_7,ae_8,ae_9,ae_10,ae_11,ae_12,ae_13
0,1,-4.406366,-1.893490,-4.860766,10.483939,-2.952837,-6.017283,1.625875,5.934033,1.859101,-1.943249,1.329634,-1.781883,1.263428
1,2,-1.525542,2.367595,-1.534571,8.508888,-4.085185,-7.362682,-2.897106,2.801864,2.670520,2.836757,1.098903,-0.138598,-0.802121
2,3,-1.760156,0.054254,-0.914767,0.844683,-1.185236,-5.744865,4.125343,2.357876,2.513872,-3.528446,-1.466366,-3.248106,-2.132906
3,4,-1.205828,4.405512,2.952558,-2.593609,-1.653408,-6.934993,1.503002,-3.399956,1.053031,0.764881,3.978532,-0.204923,-2.822405
4,5,-2.604524,1.899337,-2.099546,3.412642,-3.847823,-2.465474,2.383733,5.084182,3.629400,-0.279647,-0.533422,-1.083439,2.463400


## How to use this in Week 7

If the autoencoder is competitive or better than PCA, the exported embeddings can replace PCA as the input to the Week 7 K-means sweep.

If PCA still wins, this notebook still gives a clean justification for staying with PCA and avoids guesswork.